<a href="https://colab.research.google.com/github/LucasFaleirosMoreira/university-studies/blob/main/IA_01_11_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**EQUIPE:**

Murilo Berti

Grabriel Morais

Lucas Faleiros

In [ ]:
# Instalar dependências necessárias
!pip install nltk scikit-learn pandas tqdm

# Importações principais
import pandas as pd
import numpy as np
import os
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm


In [ ]:
# Baixar e extrair o dataset
!wget -q https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xzf aclImdb_v1.tar.gz


In [ ]:
# Download e extração do dataset IMDb
!wget -q https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xzf aclImdb_v1.tar.gz


# Função para carregar o dataset
def load_imdb_data(base_path="aclImdb"):
    data = {"text": [], "label": []}
    for label in ["pos", "neg"]:
        folder = os.path.join(base_path, "train", label)
        for file in os.listdir(folder):
            with open(os.path.join(folder, file), "r", encoding="utf-8") as f:
                data["text"].append(f.read())
                data["label"].append(1 if label == "pos" else 0)
    return pd.DataFrame(data)

df = load_imdb_data()
print("Amostra dos dados:")
print(df.head())


# Pré-processamento do texto
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    # 1. Minúsculas
    text = text.lower()
    # 2. Remover caracteres não alfabéticos
    text = ''.join([c if c.isalpha() or c.isspace() else ' ' for c in text])
    # 3. Tokenização
    tokens = word_tokenize(text)
    # 4. Remover stopwords
    tokens = [t for t in tokens if t not in stop_words]
    # 5. Lematização
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

tqdm.pandas()
df['clean_text'] = df['text'].progress_apply(preprocess_text)

print("Pré-processamento concluído com sucesso!")
print(df[['text', 'clean_text', 'label']].head())


# Vetorização com TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X = vectorizer.fit_transform(df['clean_text'])
y = df['label']


# Treino do modelo de Regressão Logística
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


# Avaliação do modelo
print("Relatório de classificação:")
print(classification_report(y_test, y_pred, target_names=["Negativo", "Positivo"]))


# Salvando modelo e vetorizador
joblib.dump(model, "modelo_imdb.pkl")
joblib.dump(vectorizer, "vetorizador_imdb.pkl")
print("Modelos salvos com sucesso!")


Amostra dos dados:
                                                text  label
0  Enjoyed the movie very much. Certainly will le...      1
1  I have always been a huge fan of "Homicide: Li...      1
2  Sweet romantic drama/comedy about Stewart and ...      1
3  One of my favorite films for a number of years...      1
4  I really like this film... when I started to w...      1


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
100%|██████████| 25000/25000 [00:43<00:00, 580.80it/s]


Pré-processamento concluído com sucesso!
                                                text  \
0  Enjoyed the movie very much. Certainly will le...   
1  I have always been a huge fan of "Homicide: Li...   
2  Sweet romantic drama/comedy about Stewart and ...   
3  One of my favorite films for a number of years...   
4  I really like this film... when I started to w...   

                                          clean_text  label  
0  enjoyed movie much certainly leave audience wa...      1  
1  always huge fan homicide life street heard reu...      1  
2  sweet romantic drama comedy stewart sullavan w...      1  
3  one favorite film number year last action hero...      1  
4  really like film started watch thought would g...      1  
Relatório de classificação:
              precision    recall  f1-score   support

    Negativo       0.88      0.85      0.86      2485
    Positivo       0.86      0.88      0.87      2515

    accuracy                           0.87      5000
   m